# Q-learning

In this demonstration, we'll go through a complete implementation of [Q-learning](https://en.wikipedia.org/wiki/Q-learning). This is a state value based method that maintains a table of values for state-action pairs.

This table is optimized by an algorithm close to the classical [temporal difference learning](https://en.wikipedia.org/wiki/Temporal_difference_learning), with the major difference that we optimize $Q(s, a)$ and not $V(s)$ (the expected return after having performed the action $a$ in the state $s$ compared to the expected return in state $s$).

## Imports

In [1]:
from collections.abc import Hashable, Sequence
from io import StringIO
from itertools import product
from random import choice as random_choice
from random import random
from typing import Any, Protocol, cast

import gymnasium as gym
from numpy import float32, linspace, zeros

## Types

Those types are completely optional: just here to document the expected arguments in the functions in the rest of the demonstration.

In [2]:
class BaseEnvironment[StateT: Hashable, ActionT: Hashable](Protocol):
  def step(
    self, action: ActionT
  ) -> tuple[StateT, float, bool, bool, dict[Any, Any]]: ...

  def reset(self) -> tuple[StateT, dict[str, Any]]: ...

## Q-learning

3 important functions:

- `bellman_update` implements the temporal difference update
- `episode` implements the sequence of `bellman_update`s of an episode
- `learn` implements the sequence of `episode`s of a complete training

In [3]:
class QLearner[StateT: Hashable, ActionT: Hashable]:
  def __init__(
    self,
    actions: Sequence[ActionT],
    states: Sequence[StateT],
    environment: BaseEnvironment[StateT, ActionT],
    gamma: float = 0.9,
    alpha: float = 0.1,
    n_iter: int = 100,
  ):
    self.actions = actions
    self.action_to_index = {a: i for i, a in enumerate(actions)}
    self.states = states
    self.state_to_index = {s: i for i, s in enumerate(states)}
    self.environment = environment
    self.gamma = gamma
    self.alpha = alpha
    self.n_iter = n_iter
    self.q_table = zeros((len(states), len(actions)), dtype=float32)

  def __str__(self) -> str:
    with StringIO() as string_io:
      actions_str = " | ".join(f"{a!s:^5}" for a in self.actions)
      string_io.write(f"{' ' * 10} | {actions_str}\n")
      for state, row in zip(self.states, self.q_table):
        state_values = " | ".join(f"{v:5.2f}" for v in row)
        string_io.write(f"{state!s:10} | {state_values}\n")
      return string_io.getvalue()

  def learn(self) -> float:
    len_i = len(str(self.n_iter))
    for i, epsilon in enumerate(linspace(1, 0, self.n_iter), start=1):
      self.epsilon = epsilon
      reward = self.episode()
      print(f"iteration {i:{len_i}}, ε = {epsilon:.2f}, r = {reward:7.2f}")
    return reward

  def get_q(self, state: StateT, action: ActionT) -> float:
    return self.q_table[self.state_to_index[state], self.action_to_index[action]]

  def set_q(self, state: StateT, action: ActionT, value: float) -> None:
    self.q_table[self.state_to_index[state], self.action_to_index[action]] = value

  def bellman_update(self, s: StateT, a: ActionT, r: float, new_s: StateT) -> None:
    q = self.get_q(s, a)
    best_new_s_a = self.best_action(new_s)
    new_s_max_q = self.get_q(new_s, best_new_s_a)
    self.set_q(s, a, q + self.alpha * (r + self.gamma * new_s_max_q - q))

  def best_action(self, s: StateT) -> ActionT:
    _, a = max((self.get_q(s, a), a) for a in self.actions)
    return a

  def episode(self) -> float:
    s, _ = self.environment.reset()
    total_r = 0.0
    done = False
    while not done:
      if random() >= self.epsilon:
        # Exploitation
        a = self.best_action(s)
      else:
        # Exploration
        a = random_choice(self.actions)
      new_s, r, done, _, _ = self.environment.step(a)
      total_r += r
      self.bellman_update(s, a, r, new_s)
      s = new_s
    return total_r

## Usage

To use a Q-learning algorithm, you need to define an environment, in particular its `step` function, which takes the agent's action and returns the new state, the reward, and whether the episode is over.

In [4]:
class Environment:
  def __init__(self) -> None:
    self.allowed = frozenset(
      {
        (0, 1),
        (0, 3),
        (0, 4),
        (0, 5),
        (1, 1),
        (1, 3),
        (2, 1),
        (2, 2),
        (2, 3),
        (2, 4),
        (2, 5),
        (2, 6),
        (3, 1),
        (4, 1),
      }
    )

  def reset(self) -> tuple[tuple[int, int], dict[str, Any]]:
    self.current_state = (2, 0)
    return self.current_state, {}

  def step(
    self, action: str
  ) -> tuple[tuple[int, int], float, bool, bool, dict[Any, Any]]:
    r, c = self.current_state
    if action == "↑":
      r -= 1
    elif action == "→":
      c += 1
    elif action == "↓":
      r += 1
    elif action == "←":
      c -= 1
    else:
      raise ValueError("Invalid action")
    if 0 <= r < 5 and 1 <= c < 6 and (r, c) in self.allowed:
      self.current_state = (r, c)
      return (r, c), -1.0, False, False, {}
    if (r, c) == (2, 6):
      self.current_state = (r, c)
      return (r, c), -1.0, True, False, {}
    return self.current_state, -1.0, False, False, {}

In [5]:
def test_qlearn() -> None:
  actions = ["↑", "→", "↓", "←"]
  states = [(i, j) for i, j in product(range(5), range(1, 6))] + [(2, 0), (2, 6)]

  environment = Environment()

  q_learner: QLearner[tuple[int, int], str] = QLearner(
    actions=actions, states=states, environment=environment, n_iter=1000
  )
  final_reward = q_learner.learn()
  assert int(final_reward) == -6
  print(q_learner)

In [6]:
test_qlearn()

iteration    1, ε = 1.00, r = -484.00
iteration    2, ε = 1.00, r = -318.00
iteration    3, ε = 1.00, r = -384.00
iteration    4, ε = 1.00, r =  -53.00
iteration    5, ε = 1.00, r = -244.00
iteration    6, ε = 0.99, r =  -75.00
iteration    7, ε = 0.99, r = -103.00
iteration    8, ε = 0.99, r = -105.00
iteration    9, ε = 0.99, r =  -44.00
iteration   10, ε = 0.99, r = -288.00
iteration   11, ε = 0.99, r =  -34.00
iteration   12, ε = 0.99, r = -160.00
iteration   13, ε = 0.99, r = -125.00
iteration   14, ε = 0.99, r = -261.00
iteration   15, ε = 0.99, r = -190.00
iteration   16, ε = 0.98, r = -128.00
iteration   17, ε = 0.98, r =  -75.00
iteration   18, ε = 0.98, r = -132.00
iteration   19, ε = 0.98, r = -204.00
iteration   20, ε = 0.98, r = -214.00
iteration   21, ε = 0.98, r = -136.00
iteration   22, ε = 0.98, r =  -59.00
iteration   23, ε = 0.98, r =  -97.00
iteration   24, ε = 0.98, r =  -89.00
iteration   25, ε = 0.98, r =   -9.00
iteration   26, ε = 0.97, r =  -19.00
iteration   

In [7]:
def test_frozen() -> None:
  env = cast("BaseEnvironment[int, int]", gym.make("FrozenLake-v1"))
  q_learner = QLearner(
    actions=[0, 1, 2, 3], states=list(range(16)), environment=env, n_iter=10000
  )
  final_reward = q_learner.learn()
  print(final_reward, q_learner)

In [8]:
test_frozen()

iteration     1, ε = 1.00, r =    0.00
iteration     2, ε = 1.00, r =    0.00
iteration     3, ε = 1.00, r =    0.00
iteration     4, ε = 1.00, r =    0.00
iteration     5, ε = 1.00, r =    0.00
iteration     6, ε = 1.00, r =    0.00
iteration     7, ε = 1.00, r =    0.00
iteration     8, ε = 1.00, r =    0.00
iteration     9, ε = 1.00, r =    0.00
iteration    10, ε = 1.00, r =    0.00
iteration    11, ε = 1.00, r =    0.00
iteration    12, ε = 1.00, r =    0.00
iteration    13, ε = 1.00, r =    0.00
iteration    14, ε = 1.00, r =    0.00
iteration    15, ε = 1.00, r =    0.00
iteration    16, ε = 1.00, r =    0.00
iteration    17, ε = 1.00, r =    0.00
iteration    18, ε = 1.00, r =    0.00
iteration    19, ε = 1.00, r =    0.00
iteration    20, ε = 1.00, r =    0.00
iteration    21, ε = 1.00, r =    0.00
iteration    22, ε = 1.00, r =    0.00
iteration    23, ε = 1.00, r =    0.00
iteration    24, ε = 1.00, r =    0.00
iteration    25, ε = 1.00, r =    0.00
iteration    26, ε = 1.00